# Master — analiza danych w Pythonie (szkielet wykładu)

Ten notebook ma być **główną osią 6-godzinnego wykładu**.  
Nie jest przeglądem wszystkich funkcji, tylko prowadzi po jednym workflow:

**pytanie → dane → czyszczenie → wybór/filtr → agregacja → wizualizacja → interpretacja**

## Trzy case'y
1. **Student data** — mały, kontrolowany case.
2. **Maratończycy** — `groupby`, `pivot_table`, wykresy.
3. **Ankieta studentów** — realne dane grupy.

## Notacja dla prowadzącego
- **LIVE** — kodujemy na żywo.
- **POKAŻ GOTOWE** — wystarczy uruchomić i omówić.
- **TODO DLA STUDENTÓW** — miejsce na krótkie zadanie.
- **OPCJONALNE** — tnij, jeśli czas siada.


## Zakładana struktura plików

```text
master_wyklad_analiza_danych_szkielet.ipynb
data/
    Ankieta_ML.csv
    marathon-data.csv   # TODO: podmień na właściwy plik
plots/
    01_domain_summary.png
    02_top_dispersion.png
    03_key_contrasts.png
    04_feedback_counts.png
```

Jeśli wolisz, możesz umieścić notebook w folderze `wyklad_2_iteracja2` i zachować te same ścieżki względne.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Image, Markdown
except Exception:
    display = print
    Image = None
    Markdown = None

BASE = Path(".")
DATA_DIR = BASE / "data"
PLOTS_DIR = BASE / "plots"

SURVEY_PATH = DATA_DIR / "Ankieta_ML.csv"
MARATHON_PATH = DATA_DIR / "marathon-data.csv"   # TODO: podmień na właściwy plik

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("BASE =", BASE.resolve())
print("DATA_DIR exists:", DATA_DIR.exists())
print("PLOTS_DIR exists:", PLOTS_DIR.exists())

## Agenda wykładu

1. **Motywacja:** po co analiza danych i co mówi ankieta.
2. **Case 1:** mały, kontrolowany dataset studentów.
3. **Case 2:** maratończycy i agregacje.
4. **Case 3:** ankieta grupy.
5. **Pułapki:** `NaN`, `merge`, `groupby` vs `pivot_table`, stringi.
6. **Mini-zadanie końcowe.**


# 0. Motywacja — zaczynamy od pytań, nie od funkcji

**Cel:** pokazać studentom, że dziś nie uczą się listy metod, tylko sposobu pracy z danymi.

### Co powiesz
- Mamy własne dane grupy.
- Chcemy umieć wyciągać z nich sensowne wnioski.
- Python ma nam w tym pomóc, a nie być celem samym w sobie.

**POKAŻ GOTOWE:** 3–4 wykresy z ankiety, ale bez pełnego live codingu od pierwszej minuty.


In [ ]:
# POKAŻ GOTOWE — jeśli pliki istnieją, pokaż je na starcie
plot_files = [
    "01_domain_summary.png",
    "02_top_dispersion.png",
    "03_key_contrasts.png",
    "04_feedback_counts.png",
]

for fname in plot_files:
    path = PLOTS_DIR / fname
    if path.exists() and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"[brak pliku lub brak IPython.display] {path}")

## Oś pracy na danych

Na każdym case'ie wracamy do tej samej sekwencji:

1. Co chcemy wiedzieć?
2. Jak wyglądają dane?
3. Co trzeba wyczyścić?
4. Jak wybrać interesujące rekordy?
5. Jak policzyć agregat?
6. Jak to pokazać na wykresie?
7. Co z tego wynika?


# 1. Case 1 — Student data (kontrolowany case)

**Cel dydaktyczny:** przejść od prostych list / tablic do DataFrame i pokazać pełny, mały workflow.

### LIVE
- lista i `numpy.array`
- `shape`, `mean`
- zbudowanie `DataFrame`
- `isna`, `fillna`, `dropna`
- filtr
- `groupby`
- prosty wykres


In [ ]:
# LIVE — mały dataset do rozgrzewki
names = [
    "Dan", "Joann", "Pedro", "Rosie", "Ethan", "Vicky", "Frederic",
    "Jimmie", "Rhonda", "Giovanni", "Francesca", "Rajab", "Naiyana",
    "Kian", "Jenny", "Jakeem", "Helena", "Ismat", "Anila", "Skye",
    "Daniel", "Aisha"
]

grades = np.array([50, 50, 47, 97, 49, 3, 52, 42, 26, 74, 82, 62, 37, 15, 70, 27, 36, 35, 48, 52, 63, 64])
study_hours = np.array([10.0, 11.5, 9.0, 16.0, 9.25, 1.0, 11.5, 9.0, 8.5, 14.5, 15.5,
                        13.75, 9.0, 8.0, 15.5, 8.0, 9.0, 6.0, 10.0, 12.0, 12.5, 12.0])

student_data = np.array([study_hours, grades])

print("shape:", student_data.shape)
print("avg study:", student_data[0].mean())
print("avg grade:", student_data[1].mean())

In [ ]:
# LIVE — przejście do pandas
df_students = pd.DataFrame({
    "Name": names,
    "StudyHours": study_hours,
    "Grade": grades,
})

# Symulujemy braki danych, żeby pokazać typowy workflow
df_students.loc[2, "StudyHours"] = np.nan
df_students.loc[5, "Grade"] = np.nan

df_students.head()

In [ ]:
# LIVE — ogląd danych i braki
print(df_students.shape)
display(df_students.head())
display(df_students.isna().sum())

# TODO: pokaż .loc, .iloc, maskę, query


In [ ]:
# LIVE — czyszczenie
df_students["StudyHours"] = df_students["StudyHours"].fillna(df_students["StudyHours"].mean())
df_students = df_students.dropna(axis=0).copy()

df_students["Passed"] = df_students["Grade"] >= 60

display(df_students.head())
display(df_students.isna().sum())

In [ ]:
# LIVE — proste pytania analityczne
mean_study = df_students["StudyHours"].mean()
mean_grade = df_students["Grade"].mean()

print(f"Średnia liczba godzin nauki: {mean_study:.2f}")
print(f"Średnia ocena: {mean_grade:.2f}")

above_mean = df_students[df_students["StudyHours"] > mean_study]
display(above_mean)

print("Średnia ocena w grupie powyżej średniej liczby godzin:",
      above_mean["Grade"].mean())

In [ ]:
# LIVE — groupby
group_pass = (
    df_students
    .groupby("Passed")[["StudyHours", "Grade"]]
    .mean()
)

display(group_pass)

In [ ]:
# LIVE — prosty wykres
group_pass["Grade"].plot(kind="bar", rot=0, title="Średnia ocena: Pass/Fail")
plt.ylabel("Grade")
plt.show()

### TODO DLA STUDENTÓW — 5 minut

1. Policz medianę `Grade`.
2. Wybierz studentów z `StudyHours > 12`.
3. Policz średnią `StudyHours` osobno dla `Passed=True/False`.
4. Narysuj histogram ocen albo bar plot dla `Passed`.

> Oddają jedną komórkę kodu i jedno zdanie interpretacji.


# 2. Case 2 — Maratończycy

**Cel dydaktyczny:** `groupby`, `pivot_table`, prosta interpretacja, połączenie z intuicją SQL.

### Co chcesz, żeby zapamiętali
- `WHERE` ≈ maska
- `GROUP BY` ≈ `groupby`
- `JOIN` ≈ `merge`
- `pivot_table` daje szybki przekrój 2D


In [ ]:
# LIVE / POKAŻ GOTOWE — wczytanie danych
if MARATHON_PATH.exists():
    df_marathon = pd.read_csv(MARATHON_PATH)
    print("shape:", df_marathon.shape)
    display(df_marathon.head())
else:
    warnings.warn(
        f"Brak pliku {MARATHON_PATH}. Podmień ścieżkę albo skopiuj dane do data/."
    )

### Adapter danych

Ten blok trzeba dopasować do rzeczywistego pliku.  
W najprostszym wariancie potrzebujesz kolumn typu:

- `age`
- `gender`
- `split`
- `final`

Jeśli masz inny plik, zrób tutaj adapter do wspólnego formatu.


In [ ]:
# TODO — dopasuj do realnego pliku maratońskiego
# Wersja przykładowa dla kolumn: age, gender, split, final

if "df_marathon" in globals():
    required_cols = {"age", "gender", "split", "final"}
    if required_cols.issubset(df_marathon.columns):
        work = df_marathon.copy()

        for col in ["split", "final"]:
            work[col + "_td"] = pd.to_timedelta(work[col])
            work[col + "_min"] = work[col + "_td"].dt.total_seconds() / 60

        work["slowdown_min"] = work["final_min"] - 2 * work["split_min"]

        age_bins = [15, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 100]
        age_labels = ["16-19", "20-24", "25-29", "30-34", "35-39", "40-44",
                      "45-49", "50-54", "55-59", "60-64", "65-69", "70+"]
        work["age_group"] = pd.cut(work["age"], bins=age_bins, labels=age_labels)

        display(work.head())
    else:
        print("TODO: zrób adapter do wspólnego formatu. Brakuje kolumn:", required_cols - set(df_marathon.columns))

In [ ]:
# LIVE — groupby po grupie wiekowej
if "work" in globals():
    age_summary = (
        work.groupby("age_group", observed=False)
            .agg(
                n=("final_min", "size"),
                mean_final=("final_min", "mean"),
                median_final=("final_min", "median"),
                mean_split=("split_min", "mean"),
                mean_slowdown=("slowdown_min", "mean"),
            )
            .reset_index()
    )
    display(age_summary.round(2))

In [ ]:
# LIVE — pivot_table: grupa wiekowa x płeć
if "work" in globals():
    pivot_mean = pd.pivot_table(
        work,
        index="age_group",
        columns="gender",
        values="final_min",
        aggfunc="mean",
        observed=False,
    )
    display(pivot_mean.round(2))

In [ ]:
# LIVE — wykres 1
if "age_summary" in globals():
    plt.figure(figsize=(10, 4))
    plt.plot(age_summary["age_group"].astype(str), age_summary["mean_final"], marker="o", label="mean_final")
    plt.plot(age_summary["age_group"].astype(str), age_summary["median_final"], marker="o", label="median_final")
    plt.xticks(rotation=35)
    plt.ylabel("minuty")
    plt.title("Czas końcowy vs grupa wiekowa")
    plt.legend()
    plt.show()

In [ ]:
# LIVE / OPCJONALNE — wykres 2
if "pivot_mean" in globals():
    pivot_mean.round(2).plot(kind="bar", figsize=(10, 4))
    plt.ylabel("średni czas końcowy [min]")
    plt.title("Średni wynik końcowy: grupa wiekowa × płeć")
    plt.show()

### TODO DLA STUDENTÓW — 8 minut

1. Policz `pivot_table` dla `slowdown_min`.
2. Posortuj `age_summary` po `mean_slowdown`.
3. Narysuj własny wykres dla jednej zależności.

> Tu możesz bardzo krótko połączyć `groupby` i `pivot_table` z SQL-em.


# 3. Case 3 — Ankieta studentów

**Cel dydaktyczny:** zastosować ten sam workflow do realnych danych grupy.

### Ważna decyzja
Nie rób pełnej analizy wszystkiego live.  
Na żywo pokaż tylko:
- wczytanie,
- konwersję do liczb,
- prostą agregację,
- jeden wykres,
- jedną interpretację.

Resztę pokaż jako gotowe wyniki i omów.


In [ ]:
# LIVE — wczytanie ankiety
if SURVEY_PATH.exists():
    df_raw = pd.read_csv(SURVEY_PATH)
    timestamp_col = df_raw.columns[0]
    questions = df_raw.columns[1:].tolist()

    df = df_raw.copy()
    for c in questions:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # W pytaniach feedbackowych 0 = "nie było mnie"
    df_no0 = df.copy()
    for c in questions:
        df_no0[c] = df_no0[c].replace(0, np.nan)

    print("Liczba odpowiedzi:", len(df_no0))
    print("Liczba pytań:", len(questions))
    display(df_no0.head())
else:
    warnings.warn(f"Brak pliku {SURVEY_PATH}")

In [ ]:
# LIVE — mapowanie pytań na domeny
if "questions" in globals():
    section_map = {}
    for i in range(1, 65):
        if 1 <= i <= 8:
            sec = "Python / programowanie"
        elif 9 <= i <= 15:
            sec = "NumPy / pandas / wizualizacja"
        elif 16 <= i <= 19:
            sec = "SQL / relacyjne bazy"
        elif 20 <= i <= 31:
            sec = "Matematyka i statystyka"
        elif 32 <= i <= 42:
            sec = "ML podstawy + sprawność algorytmiczna"
        else:
            sec = "Praca, preferencje i feedback"
        section_map[i] = sec

    qmap = pd.DataFrame({
        "qnum": range(1, 65),
        "question": questions,
        "section": [section_map[i] for i in range(1, 65)],
    })

    display(qmap.head(20))

In [ ]:
# LIVE — prosta tabela statystyk pytań
if "questions" in globals():
    question_stats = pd.DataFrame({
        "qnum": range(1, 65),
        "question": questions,
        "section": [section_map[i] for i in range(1, 65)],
        "mean": [df_no0[c].mean() for c in questions],
        "median": [df_no0[c].median() for c in questions],
        "std": [df_no0[c].std() for c in questions],
        "pct_4_5": [(df_no0[c] >= 4).mean() * 100 for c in questions],
        "n_valid": [df_no0[c].notna().sum() for c in questions],
    })

    display(question_stats.sort_values("mean").head(10))

In [ ]:
# LIVE — średnie domenowe
if "questions" in globals():
    domain_defs = {
        "Python / programowanie": list(range(1, 9)),
        "NumPy / pandas / wizualizacja": list(range(9, 16)),
        "SQL / relacyjne bazy": list(range(16, 20)),
        "Matematyka i statystyka": list(range(20, 32)),
        "ML podstawy + sprawność algorytmiczna": list(range(32, 43)),
        "Praca, preferencje i feedback": list(range(43, 65)),
    }

    domain_rows = []
    for name, idxs in domain_defs.items():
        cols = [questions[i - 1] for i in idxs]
        student_means = df_no0[cols].mean(axis=1)
        domain_rows.append({
            "domain": name,
            "mean": student_means.mean(),
            "median": student_means.median(),
            "std_between_students": student_means.std(),
            "n_questions": len(cols),
        })

    domain_summary = pd.DataFrame(domain_rows).sort_values("mean")
    display(domain_summary)

In [ ]:
# LIVE — jeden wykres na żywo
if "domain_summary" in globals():
    plt.figure(figsize=(10, 5.8))
    plt.barh(
        domain_summary["domain"],
        domain_summary["mean"],
        xerr=domain_summary["std_between_students"],
        capsize=4,
    )
    plt.xlim(0, 5)
    plt.xlabel("Średnia samoocena (1–5)")
    plt.title("Średnie domenowe i zróżnicowanie między studentami")
    plt.tight_layout()
    plt.show()

### POKAŻ GOTOWE — kontrasty i rozkłady

Tu nie musisz kodować wszystkiego.  
Wystarczy pokazać gotowe wykresy i powiedzieć, jakie pytania są ważniejsze niż pojedyncze średnie.


In [ ]:
# POKAŻ GOTOWE — helper do porównywania rozkładów wybranych pytań
def plot_question_distributions(df_full, questions, qnums, title=None):
    fig, axes = plt.subplots(1, len(qnums), figsize=(4 * len(qnums), 3.8), sharey=True)
    if len(qnums) == 1:
        axes = [axes]
    for ax, q in zip(axes, qnums):
        s = df_full[questions[q - 1]].replace(0, np.nan).value_counts().sort_index()
        xs = [1, 2, 3, 4, 5]
        ax.bar(xs, [s.get(i, 0) for i in xs])
        ax.set_title(f"Q{q}: " + questions[q - 1][:38] + "...")
        ax.set_xticks(xs)
        ax.set_xlabel("Ocena")
    axes[0].set_ylabel("Liczba odpowiedzi")
    if title:
        plt.suptitle(title)
        plt.tight_layout(rect=[0, 0, 1, 0.92])
    else:
        plt.tight_layout()
    plt.show()

# PRZYKŁADY do użycia na żywo:
# plot_question_distributions(df, questions, [12, 17], "pandas groupby vs SQL GROUP BY")
# plot_question_distributions(df, questions, [13, 18], "pandas merge vs SQL JOIN")
# plot_question_distributions(df, questions, [50, 51, 52, 53], "pisemne, projekty, teoria i praktyka")

### TODO DLA STUDENTÓW — 8 minut

Wybierz jedno:
1. znajdź 5 pytań o najniższej średniej,
2. znajdź 5 pytań o największym rozrzucie,
3. porównaj 2 pytania, które Twoim zdaniem powinny być omówione razem.

> Oddają tabelę albo jeden wykres plus jedno zdanie interpretacji.


# 4. Pułapki i szczególne przypadki

Ten blok ma być krótki.  
Nie kolejny wykład, tylko lista rzeczy, które na realnych danych często psują wynik.


In [ ]:
# 4.1 None vs NaN
demo = pd.DataFrame({
    "a": [1, 2, None, 4],
    "b": [1.0, np.nan, 3.5, 4.5],
    "c": ["x", None, "z", "w"],
})
display(demo)
display(demo.isna().sum())

In [ ]:
# 4.2 groupby vs pivot_table
demo_gp = pd.DataFrame({
    "group": ["A", "A", "B", "B", "B"],
    "gender": ["K", "M", "K", "M", "K"],
    "score": [5, 7, 6, 8, 9],
})

display(demo_gp.groupby("group")["score"].mean())

display(pd.pivot_table(
    demo_gp,
    index="group",
    columns="gender",
    values="score",
    aggfunc="mean",
))

In [ ]:
# 4.3 merge i zła granularność
left = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Ala", "Bartek", "Celina"],
})

right = pd.DataFrame({
    "id": [1, 1, 2, 3],
    "score": [10, 11, 20, 30],
})

display(left)
display(right)

bad_merge = left.merge(right, on="id", how="left")
display(bad_merge)

# TODO: pokaż validate='one_to_many' albo wyjaśnij, czemu liczba wierszy rośnie

In [ ]:
# 4.4 Stringi w pandas
demo_str = pd.DataFrame({
    "record": ["Kowalski;M;35-39;03:42:15", "Nowak;K;25-29;03:21:10"]
})

parts = demo_str["record"].str.split(";", expand=True)
parts.columns = ["name", "gender", "age_group", "time"]
display(parts)

# 5. Mini-zadanie końcowe

## Poziom podstawowy
- wczytaj dane,
- wybierz podzbiór,
- policz średnią,
- pokaż jeden wykres.

## Poziom standardowy
- `groupby` albo `pivot_table`,
- prosty wykres,
- jedno zdanie interpretacji.

## Poziom rozszerzony
- `merge` albo porównanie dwóch grup / dwóch pytań,
- własny wniosek.


# 6. Podsumowanie

Na koniec studenci mają zapamiętać nie listę funkcji, tylko jeden sposób pracy:

1. Zadaj pytanie.
2. Obejrzyj dane.
3. Wyczyść to, co przeszkadza.
4. Wybierz interesujący fragment.
5. Policz agregat.
6. Pokaż wynik.
7. Powiedz, co z niego wynika.

## 5 narzędzi, które muszą pamiętać
- `read_csv`
- filtrowanie maską
- `isna` / `fillna` / `dropna`
- `groupby`
- `pivot_table`
- prosty wykres w `matplotlib` albo `plot`

## Co możesz zadać po zajęciach
- mała analiza własnego CSV,
- 1 wykres,
- 1 tabela,
- 3 zdania wniosków.
